In [ ]:
! pip install transformers datasets
pip install evaluate

In [ ]:
def compute_metrics(eval_preds):
    logits, labels = eval_preds
    preds = np.argmax(logits, axis=-1)

    exact_matches = 0
    total = 0

    for pred_ids, label_ids in zip(preds, labels):
        label_ids = label_ids[label_ids != -100]
        pred_ids = pred_ids[:len(label_ids)]
        pred_text = tokenizer.decode(pred_ids, skip_special_tokens=True).strip()
        label_text = tokenizer.decode(label_ids, skip_special_tokens=True).strip()

        if pred_text == label_text:
            exact_matches += 1
        total += 1

    return {"exact_match": exact_matches / total}

In [ ]:
import torch
from transformers import PreTrainedTokenizerFast, Trainer, TrainingArguments, DataCollatorForLanguageModeling
from datasets import load_dataset
from transformers import DeepseekV3ForCausalLM  # Deepseek architecture

# -------------------------------
# Paths & Hyperparameters
# -------------------------------
model_path = "/kaggle/input/data-pretrained-model"
tokenizer_path = "/kaggle/input/tokeniser/multilingual_bpe_64k.json"

train_file = "/kaggle/input/task-splits/reverse_train.jsonl"
val_file   = "/kaggle/input/task-splits/reverse_val.jsonl"
test_file  = "/kaggle/input/task-splits/reverse_test.jsonl"

max_length = 96
response_max_length = 32

# -------------------------------
# Load tokenizer
# -------------------------------
tokenizer = PreTrainedTokenizerFast(tokenizer_file=tokenizer_path)
tokenizer.eos_token = "</s>"
tokenizer.pad_token = "<pad>"
tokenizer.padding_side = "left"

# -------------------------------
# Load pretrained Deepseek model
# -------------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
model = DeepseekV3ForCausalLM.from_pretrained(model_path, trust_remote_code=True)
model.to(device)
model.train()

# -------------------------------
# Load dataset
# -------------------------------
dataset = load_dataset("json", data_files={
    "train": train_file,
    "validation": val_file,
    "test": test_file
})

# -------------------------------
# Preprocess dataset
# -------------------------------
def preprocess(example):
    prompt = f"User: {example['input']}\nAssistant:"
    target = example['output']
    full_text = prompt + " " + target + tokenizer.eos_token

    tokenized = tokenizer(
        full_text,
        max_length=max_length,
        truncation=True,
        padding="max_length"
    )

    # Mask prompt tokens in labels
    prompt_len = len(tokenizer(prompt)["input_ids"])
    labels = tokenized["input_ids"].copy()
    labels[:prompt_len] = [-100] * prompt_len  # use list of -100
    tokenized["labels"] = labels

    return tokenized


tokenized_dataset = dataset.map(preprocess, batched=False)

# -------------------------------
# Data collator
# -------------------------------
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# -------------------------------
# Training arguments
# -------------------------------
training_args = TrainingArguments(
    output_dir="./reverse_finetune",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=10,
    learning_rate=5e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=100,
    fp16=True,
    report_to="none"
)

# -------------------------------
# Trainer
# -------------------------------
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
    tokenizer=tokenizer
    compute_metrics=compute_metrics
)

# -------------------------------
# Train
# -------------------------------
trainer.train()


In [ ]:

# # Test set (same fix)
# print("Test set predictions:")
# for example in dataset["test"]:
#     prompt = f"User: {example['input']}\nAssistant:"
#     inputs = tokenizer(prompt, return_tensors="pt").to(device)

#     if "token_type_ids" in inputs:
#         inputs.pop("token_type_ids")

#     outputs = model.generate(
#         **inputs,
#         max_new_tokens=response_max_length,
#         do_sample=False
#     )

#     prompt_len = inputs["input_ids"].shape[1]
#     generated_ids = outputs[0][prompt_len:]
#     reversed_word = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

#     print(f"Input: {example['input']}")
#     print(f"Prediction: {reversed_word}")
#     print(f"Expected: {example['output']}\n")


In [ ]:
eval_results = trainer.evaluate(eval_dataset=tokenized_dataset["validation"])
print("Validation Perplexity:", torch.exp(torch.tensor(eval_results["eval_loss"])))


In [ ]:
model.eval()
correct = 0
total = 0

for example in dataset["test"]:
    prompt = f"User: Reverse the word: {example['input']}\nAssistant:"
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    if "token_type_ids" in inputs:  # remove for decoder-only model
        inputs.pop("token_type_ids")

    outputs = model.generate(
        **inputs,
        max_new_tokens=response_max_length,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id
    )

    # Only take generated tokens after the prompt
    prompt_len = inputs["input_ids"].shape[1]
    generated_ids = outputs[0][prompt_len:]
    pred_word = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

    if pred_word == example["output"]:
        correct += 1
    total += 1

print("Test Exact-match Accuracy:", correct / total)


In [ ]:
# Directory where you want to save
save_dir = "./finetuned_model_rev_task"

# Save the model
model.save_pretrained(save_dir)

# Save the tokenizer
tokenizer.save_pretrained(save_dir)


In [ ]:
import torch
from transformers import PreTrainedTokenizerFast, DeepseekV3ForCausalLM

# Path to saved fine-tuned model
save_dir = "./finetuned_model_rev_task"

# Load tokenizer
tokenizer = PreTrainedTokenizerFast.from_pretrained(save_dir)
tokenizer.eos_token = "</s>"
tokenizer.pad_token = "<pad>"

# Load model
device = "cuda" if torch.cuda.is_available() else "cpu"
model = DeepseekV3ForCausalLM.from_pretrained(save_dir, trust_remote_code=True)
model.to(device)
model.eval()


In [ ]:
def reverse_word_inference(word, max_new_tokens=32):
    # Prepare prompt
    prompt = f"User: Reverse the word: {word}\nAssistant:"
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    # Remove token_type_ids if present (decoder-only model)
    if "token_type_ids" in inputs:
        inputs.pop("token_type_ids")

    # Generate
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id
    )

    # Only take generated tokens after prompt
    prompt_len = inputs["input_ids"].shape[1]
    generated_ids = outputs[0][prompt_len:]
    reversed_word = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

    return reversed_word


In [ ]:
test_words = ["hello", "निबंधकर", "కవ్వగుంట", "soccer"]

for word in test_words:
    reversed_word = reverse_word_inference(word)
    print(f"Input: {word} -> Prediction: {reversed_word}")
